In [0]:
from pyspark.sql import functions as F

In [0]:
CATALOG = "opensky_lakehouse"
LANDING_SCHEMA = "landing"
BRONZE_SCHEMA = "bronze"
VOLUME = "raw_files"

RAW_BASE_PATH = f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME}/opensky/states"

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.opensky_states_raw"

In [0]:
raw_files_df = (
    spark.read
    .format("binaryFile")
    .option("pathGlobFilter", "*.json")
    .option("recursiveFileLookup", "true")
    .load(RAW_BASE_PATH)
)

display(raw_files_df)

In [0]:
bronze_df = (
    raw_files_df
    .select(
        F.col("path").alias("source_file"),
        F.col("modificationTime").alias("source_file_modification_time"),
        F.col("length").alias("source_file_size_bytes"),
        F.col("content").cast("string").alias("raw_json"),
        F.current_timestamp().alias("bronze_ingestion_timestamp"),
        F.lit("opensky").alias("source_system")
    )
    .withColumn(
        "source_date",
        F.regexp_extract(
            F.col("source_file"),
            r"date=([0-9]{4}-[0-9]{2}-[0-9]{2})",
            1
        )
    )
    .withColumn(
        "raw_hash",
        F.sha2(F.col("raw_json"), 256)
    )
)

display(bronze_df)

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")

In [0]:
table_exists = spark.catalog.tableExists(BRONZE_TABLE)

if table_exists:
    existing_hashes_df = spark.table(BRONZE_TABLE).select("raw_hash").distinct()

    bronze_new_df = (
        bronze_df
        .join(existing_hashes_df, on="raw_hash", how="left_anti")
    )
else:
    bronze_new_df = bronze_df

display(bronze_new_df)

(
    bronze_new_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Nuevos archivos insertados en Bronze: {bronze_new_df.count()}")

In [0]:
display(spark.table(BRONZE_TABLE))

In [0]:
spark.sql(f"""
SELECT
    source_system,
    source_date,
    COUNT(*) AS total_raw_files,
    MIN(bronze_ingestion_timestamp) AS first_ingestion,
    MAX(bronze_ingestion_timestamp) AS last_ingestion
FROM {BRONZE_TABLE}
GROUP BY source_system, source_date
ORDER BY source_date DESC
""").display()